In [1]:
# imports
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import KFold
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical

In [2]:
import tensorflow as tf

print("TF Version:", tf.__version__)
print("Devices:", tf.config.list_physical_devices())

TF Version: 2.10.0
Devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [3]:
class_df = pd.read_csv("E:/Datasets/AIA-image-classification/images.csv")   # change path
folder = "E:/Datasets/AIA-image-classification/images"
class_df

,image_name,label
0,0.jpg,0
1,1.jpg,4
2,2.jpg,5
3,4.jpg,0
4,7.jpg,4
...,...,...
14029,20048.jpg,0
14030,20051.jpg,1
14031,20052.jpg,3
14032,20053.jpg,4


In [4]:
image_paths = []
labels = []

for index, row in class_df.iterrows():
    image_id = str(row['image_name'])
    label = row['label']
    
    full_path = os.path.join(folder, image_id)
    
    if os.path.exists(full_path):
        image_paths.append(full_path)
        labels.append(label)

image_paths = np.array(image_paths)
labels = np.array(labels)

In [5]:
def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (224, 224))
    image = image / 255.0
    return image, label

In [ ]:
def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (224, 224))
    
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.2)
    image = tf.image.random_contrast(image, 0.8, 1.2)
    
    image = image / 255.0
    
    return image, label

In [6]:
def create_model():
    model = models.Sequential()
    
    model.add(layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)))
    model.add(layers.MaxPooling2D(2,2))
    
    model.add(layers.Conv2D(64, (3,3), activation='relu'))
    model.add(layers.MaxPooling2D(2,2))
    
    model.add(layers.Conv2D(128, (3,3), activation='relu'))
    model.add(layers.MaxPooling2D(2,2))

    model.add(layers.Conv2D(256, (3,3), activation='relu'))
    model.add(layers.MaxPooling2D(2,2))
    
    model.add(layers.Flatten())
    model.add(layers.Dense(256, activation='relu'))
    model.add(layers.Dense(6, activation='softmax'))
    
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

In [7]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

scores = []

for fold, (train_idx, val_idx) in enumerate(kfold.split(image_paths)):

    print(f"\n🔥 Fold {fold+1}")

    # Split data
    X_train, X_val = image_paths[train_idx], image_paths[val_idx]
    y_train, y_val = labels[train_idx], labels[val_idx]

    # One-hot encoding
    y_train = to_categorical(y_train, num_classes=6)
    y_val = to_categorical(y_val, num_classes=6)

    # Create datasets
    train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
    val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))

    train_ds = train_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    val_ds = val_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)

    train_ds = train_ds.shuffle(1000).batch(64).prefetch(tf.data.AUTOTUNE)
    val_ds = val_ds.batch(64).prefetch(tf.data.AUTOTUNE)

    # Build model
    model = create_model()

    # Train
    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=10
    )

    # Evaluate
    loss, acc = model.evaluate(val_ds)
    scores.append(acc)



🔥 Fold 1
Epoch 1/10
176/176 [==============================] - 23s 126ms/step - loss: 1.0611 - accuracy: 0.5803 - val_loss: 0.7766 - val_accuracy: 0.7146
Epoch 2/10
176/176 [==============================] - 22s 125ms/step - loss: 0.7112 - accuracy: 0.7359 - val_loss: 0.6627 - val_accuracy: 0.7499
Epoch 3/10
176/176 [==============================] - 22s 125ms/step - loss: 0.5610 - accuracy: 0.7943 - val_loss: 0.5741 - val_accuracy: 0.7948
Epoch 4/10
176/176 [==============================] - 22s 125ms/step - loss: 0.4481 - accuracy: 0.8390 - val_loss: 0.6252 - val_accuracy: 0.8001
Epoch 5/10
176/176 [==============================] - 22s 125ms/step - loss: 0.3427 - accuracy: 0.8741 - val_loss: 0.6272 - val_accuracy: 0.7912
Epoch 6/10
176/176 [==============================] - 22s 125ms/step - loss: 0.2356 - accuracy: 0.9171 - val_loss: 0.7472 - val_accuracy: 0.7973
Epoch 7/10
176/176 [==============================] - 22s 125ms/step - loss: 0.1709 - accuracy: 0.9377 - val_loss: 0.762

In [8]:
print("🔥 Average Accuracy:", np.mean(scores))


🔥 Average Accuracy: 0.8037624359130859
